## Upload Vectors to PostgreSQL Database

### Installing Utilities and Libraries

In [ ]:
%pip install psycopg[binary]==3.3.4 psycopg_pool==3.3.1 "databricks-sdk>=0.89.0" langchain-community==0.4.1 databricks-openai==0.17.1

### Restart the Python Environment

In [ ]:
dbutils.library.restartPython()

### Setting up the Environment

In [ ]:
from databricks.sdk import WorkspaceClient
import psycopg

# Databricks SDK uses your existing OAuth identity
w = WorkspaceClient()

# Lakebase endpoint resource name
endpoint = (
    "projects/<project-id>/"
    "branches/<branch-id>/"
    "endpoints/<endpoint-id>"
)

# Generate a short-lived OAuth credential
credential = w.postgres.generate_database_credential(
    endpoint=endpoint
)

# Generates the workspace host URL
workspace_host = w.config.host.rstrip("/")

In [ ]:
host = "LAKEBASE_HOTSNAME"
db_name = "LAKEBASE_DATABASE_NAME"
username = "LAKEBASE_USERNAME"
password = credential.token

### Create a Connection Pool

In [ ]:
from psycopg_pool import ConnectionPool

pool = ConnectionPool(
    conninfo=(
        f"host={host} "
        f"dbname={db_name} "
        f"user={username} "
        f"password={password} "
        f"sslmode=require"
    ),
    min_size=2,
    max_size=10
)

pool.wait()

print("Connection pool created successfully")

### Creating the OpenAI Client 

In [ ]:
from databricks_openai import DatabricksOpenAI

client = DatabricksOpenAI()

### Create the Document Chunker Helper Function with LangChain

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def perform_fixed_size_chunking(document, chunk_size=500, chunk_overlap=50):
    """
    Performs recursive chunking on a document with specified overlap.
    Uses RecursiveCharacterTextSplitter which tries multiple separators.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    return text_splitter.split_text(document)

### Create the Embedding Generator Helper Function

In [ ]:
def generate_embeddings(text):

    # OpenAI Request
    completion = client.embeddings.create(
        model="databricks-gte-large-en",
        input=text
    )

    return completion.data[0].embedding

### Fetch the Report Data from Table

In [ ]:
fetch_query = """
SELECT
    RecordID,
    CompanyName,
    SustainabilityReport
FROM RAG.ESG_TextData
"""

In [ ]:
from psycopg.rows import dict_row

with pool.connection() as conn:

    with conn.cursor(
        row_factory=dict_row
    ) as cur:

        cur.execute(fetch_query)

        esg_documents = cur.fetchall()

### Generate Chunks

In [ ]:
all_chunks = []

for document in esg_documents:

    chunks = perform_fixed_size_chunking(
        document["sustainabilityreport"]
    )

    for chunk in chunks:

        all_chunks.append(
            {
                "record_id":
                    document["recordid"],

                "company_name":
                    document["companyname"],

                "chunk_text":
                    chunk
            }
        )

### Generate Embeddings

In [ ]:
for chunk in all_chunks:

    chunk["embedding"] = generate_embeddings(
        chunk["chunk_text"]
    )

### Insert Chunks into Table

In [ ]:
insert_query = """
INSERT INTO RAG.ESG_Chunks
(
    RecordID,
    CompanyName,
    ChunkText,
    ChunkEmbedding
)
VALUES
(
    %s,
    %s,
    %s,
    %s
)
"""

In [ ]:
with pool.connection() as conn:

    with conn.cursor() as cur:

        for chunk in all_chunks:

            cur.execute(
                insert_query,
                (
                    chunk["record_id"],
                    chunk["company_name"],
                    chunk["chunk_text"],
                    chunk["embedding"]
                )
            )

        conn.commit()

### Create the Lakebase ANN Vector Index

In [ ]:
create_index_query = """
CREATE INDEX ON RAG.ESG_Chunks USING lakebase_ann (ChunkEmbedding vector_cosine_ops)
WITH (build_mode = 'quality');
"""

with pool.connection() as conn:

    with conn.cursor() as cur:

        cur.execute(create_index_query)

    conn.commit()

print("Lakebase ANN Index Created Successfully")

### Verify the Index

In [ ]:
verify_query = """
SELECT
    indexname,
    indexdef
FROM pg_indexes
WHERE schemaname = 'rag'
"""

In [ ]:
from psycopg.rows import dict_row

with pool.connection() as conn:

    with conn.cursor(
        row_factory=dict_row
    ) as cur:

        cur.execute(verify_query)

        indexes = cur.fetchall()

for index in indexes:

    print(index["indexname"])
    print(index["indexdef"])
    print("---------------")